# Week 6 HW: GeoPandas Practice 

### Charlotte Strom
### GEOG 592 
### 10/2/26

# 1. Complete the hierarchy:  
1. Shapely geometry  
2. GeoSeries
3. GeoDataFrame   
# 2. What type of object is returned by each expression? Write Shapely geometry, GeoSeries, GeoDataFrame, or pandas Series. 
parks.geometry -> GeoSeries: active geometry column with all spatial shapes for dataset 
parks.geometry.iloc[0] -> Shapely: individual geometry element extracted 
parks.iloc[0] -> pandas Series
parks.iloc[[0]] -> GeoDataFrame: the double brackets forces it to maintain the tabular DataFrame structure rather then allowing it to be flattened down to a row Series (the next composition, a GeoSeries)
# 3. Explain in one sentence why `parks.geometry.iloc[0]` does not behave exactly like `parks.geometry`. 
parks.geometry.iloc[0] specifies the geometry type which is returned from one cell within the parks GeoDataFrame, while parks.geometry is the geometry returned within all of the GeoSeries containing all spatial shapes for the GeoDataFrame.  
# 4. Write code to display all column names in `parks` as a regular Python list. 
parks.columns 
# 5. Write code to select only the `NAME` and `geometry` columns from `parks`. 
parks[["NAME", "geometry"]]
# 6. Write code to select the first three rows of `parks` while keeping the result as a GeoDataFrame. 
parks.head(3) OR 
parks.iloc[0:3]
# 7. Write code to inspect the geometry type of every feature in `parks`. 
parks.geometry
# 8. What does `parks.crs` tell you? 
Returns the CRS of the parks GeoDataFrame
# 9. Why can one point still produce a complete graph when you call `.plot()`? 
When you call .plot() on a point within a GeoDataFrame, GeoPandas automatically sets boundaries, standardizes symbol sizing, and generates axis scales. 
# 10. Assume `one_park = parks.geometry.iloc[0]`. What library provides the geometry object stored in  `one_park`? 
The Shapely library (because we're extracting one single cell within the GeoDataFrame to determine its geometry, which is Shapely).



stops.crs 
# EPSG:4326 
# 11. Would EPSG:4326 be a good CRS for measuring distance in feet? Explain briefly. 
No, because EPSD:4326 is WGS 84 and is in a spherical graphical coordinate system that measures distance in decimal degrees. 
# 12. Write code to create `stops_nc` by reprojecting `stops` to EPSG:2264. 
stops_nc = stops.to_crs(epsg:2264)
stops_nc
# 13. Write code to create an `area_sqft` column in `parks_nc` using polygon geometry area. 
parks_nc["area_sqft] = parks_nc.geometry.area 
parks_nc["area_sqft]
# 14. Write code to calculate the distance from every stop in `stops_nc` to a single polygon named `park_1`. 
distance_nc = stops_nc.geometry.distance(park_1)
distance_nc 
# 15. If `park_1` is a polygon, does `.distance(park_1)` measure distance to the polygon centroid? Explain. 
No, it would measure distance to the closest point on the polygon perimeter (boundary). 
# 16. Explain each 
within: Geometry A is completely inside Geometry B (none of Geometry A's points are outside of Geometry B).
contains: inverse of within; Geometry A encloses Geometry B 
intersects: Geometry A and B share any common space 
touches: Geometry A and B share borders; they touch but do not cross boundaries. 

# 17. In `gpd.sjoin(parks, urban, predicate="within")`, which geometry is being tested as within which? 
Geometry A (parks) is being tested as within Geometry B (urban). 
# 18. A park crosses an urban-area boundary. Would `within` be True? Would `intersects` be True? 
'within' would be False, because for one geometry to be within another, it must be completely inside. 'intersects' would be True because it requires the two geometries to only share a common space, including a park crossing an urban-area boundary. 
# 19. Write code that returns how many stops are inside each polygon in a gpd named `parks` that has three polygons. 
stops_nc.sjoin(parks, predicate="within")["index_right"].value_counts()
## urban_parks = gpd.sjoin(parks, urban, predicate="within") 
# 20. What is the purpose of `gpd.sjoin()`? 
gpd.sjoin() is a method which finds which GeoPandas index contains each park and urban area. It does this by joining two fields spatially rather than by a shared ID column. 
# 21. Why is a spatial join different from a regular pandas join on a common ID field? 
A spatial join merges two geospatial datasets based on their spatial relationships rather than a shared ID column. 
# 22. Write a spatial join that keeps bus stops that fall inside park polygons. Use `stops` as the left  GeoDataFrame and `parks` as the right GeoDataFrame. 
urban_areas = gpd.sjoin(stops, parks, how='inner', predicate='within')
urban_areas
# 23. After a point-in-polygon spatial join, how could you count how many bus stops were matched to each park? Describe the idea or write code. 
I would use the gdf.value_counts() method, for example:
    joined_data["park_name"].value_counts()

# 24. Why can a polygon-to-polygon `intersects` spatial join create multiple output rows if each original layer was one polygon in size?
A polygon-to-polygon intersects spatial join will never create multiple output rows if each input layer contains exactly one single-part polygon. 
# 25. You want to answer: “Which parks are within 500 feet of a bus stop?” Write how would you do this with GeoPandas  
 ## a. using a distance and apply method.  
parks_nc["near_stop"] = parks_nc.geometry.apply(lambda park: (stops_nc.geometry.distance(park) <= 500.any()))
parks_nc[parks_nc["near_stop"]]

 ## b. using buffers

 stop_buffers = stops_nc.buffer(500)
 parks_nc["near_stop"] = parks_nc.geometry.apply(lambda park: stop_buffers.inersects(park).any())
 parks_nc[parks_nc["near_stop]]

